# Dripper / MinerU-HTML Layout Clustering — Step-by-Step Tutorial

**Machine**: dgx-a100-02 (10.184.206.11)  
**Data**: `/raid/vjawa/dripper_tutorial/` — 8192 pages from 16 hosts in CC-MAIN-2025-26  
**Model**: `opendatalab/MinerU-HTML-v1.1-hunyuan0.5B-compact` (0.5B params)

### The core idea
Running LLM extraction on every Common Crawl page is expensive (~242K H100-hours per snapshot).  
Most pages on the same site share the same DOM layout.  
This pipeline:
1. **Clusters** pages by DOM structure (CPU, cheap)
2. **Runs LLM** on one representative per cluster (GPU, expensive)
3. **Propagates** the LLM's decisions to all siblings as a template (CPU, cheap)

### Sections
0. Setup  
1. Load data  
2. DOM feature extraction  
3. Layout clustering (DBSCAN)  
4. Representative selection  
5. HTML simplification  
6. LLM extraction (from baseline)  
7. Template propagation  
8. Validation (F1 vs baseline)  
9. Cost analysis

## 0. Setup

In [ ]:
%matplotlib inline
import re
import sys
import time
from collections import Counter

CURATOR_REPO = "/raid/vjawa/nemo-curator-adlr-mm/submodules/Curator"
DATA_DIR = "/raid/vjawa/dripper_tutorial"
sys.path.insert(0, CURATOR_REPO)

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq

matplotlib.rcParams["figure.dpi"] = 100

pd.set_option("display.max_colwidth", 80)


def read_parquet(path):
    """Use ParquetFile directly — avoids ParquetDataset buffer error on pyarrow 23."""
    return pq.ParquetFile(str(path)).read().to_pandas()


def coerce_html(raw):
    if isinstance(raw, bytes):
        return raw.decode("utf-8", errors="replace")
    return str(raw or "")


def convert_to_content(bindings, main_html, url=""):
    """Convert extracted main HTML to plain text via bindings.convert2content."""
    try:
        case = bindings.case_cls(bindings.input_cls(raw_html=main_html, url=url))
        case.output_data = bindings.output_cls(main_html=main_html)
        case = bindings.convert2content(case, output_format="mm_md")
        out = getattr(case, "output_data", None)
        return str(getattr(out, "main_content", "") or main_html)
    except Exception:
        return main_html


print("Setup OK")

## 1. Load Data

In [ ]:
manifest = read_parquet(f"{DATA_DIR}/layout_precompute_manifest.parquet")
print(f"Manifest: {len(manifest):,} rows, {manifest['url_host_name'].nunique()} hosts")

try:
    baseline = read_parquet(f"{DATA_DIR}/baseline_dripper_results.parquet")
    print(f"Baseline: {len(baseline):,} rows")
except Exception as e:
    baseline = None
    print(f"Baseline not available ({e.__class__.__name__}) — sections 6-8 will be skipped")
    print(
        f"  Fix: rsync -az vjawa@nb-hel-cs-001-dc-01.nvidia.com:/lustre/fsw/portfolios/"
        f"llmservice/users/vjawa/dripper_cc_main_2025_26_smoke/328281/dripper_results.parquet "
        f"{DATA_DIR}/baseline_dripper_results.parquet"
    )

print()
print(manifest["url_host_name"].value_counts().to_string())

In [ ]:
# Inspect a few raw pages
for _, row in manifest.sample(3, random_state=42).iterrows():
    html = coerce_html(row["html"])
    print(f"URL:       {row['url']}")
    print(f"Host:      {row['url_host_name']}")
    print(f"Layout ID: {row['dripper_layout_id']}")
    print(f"HTML size: {len(html):,} chars")
    print(f"Preview:   {html[:150].strip()!r}")
    print()

## 2. DOM Feature Extraction

`get_feature()` traverses the DOM tree and returns a per-depth bag of tags + class/id attributes.  
Noisy tags (`script`, `style`, `meta`) are ignored. Dynamic attributes (UUIDs, hashes) are normalised.  
Result: a compact structural fingerprint independent of page content.

In [ ]:
from nemo_curator.stages.text.experimental.dripper.stage import (
    DripperHTMLExtractionStage,
    _load_llm_web_kit_bindings,
    _load_mineru_html_bindings,
    _token_f1,
)

web = _load_llm_web_kit_bindings()
bindings = _load_mineru_html_bindings()
print("Bindings loaded")

In [ ]:
# Same host → similar features
host_rows = manifest[manifest["url_host_name"] == "hysplitbbs.arl.noaa.gov"].head(3)
print("Features from 3 pages on hysplitbbs.arl.noaa.gov (same BBS template):")
for _, row in host_rows.iterrows():
    feat = web.get_feature(coerce_html(row["html"]))
    n_layers = len(feat.get("tags", {}))
    n_tags = sum(len(v) for v in feat.get("tags", {}).values())
    print(f"  {row['url'][-70:]}")
    print(f"    layers={n_layers}  tag_entries={n_tags}")
    # Show first 2 layers
    for layer in sorted(feat["tags"])[:2]:
        print(f"    layer {layer}: {feat['tags'][layer][:5]}")
    print()

## 3. Layout Clustering

`cluster_html_struct()` runs DBSCAN within each host:
- Weighted cosine similarity: **tag weight=0.7, attr weight=0.3**
- `eps = 1 - threshold` (default threshold=0.95)
- Pages with `layout_id=-1` are DBSCAN noise (no cluster assigned)

In [ ]:
host = "scratch.mit.edu"
rows = manifest[manifest["url_host_name"] == host].head(50)
samples = []
for i, (_, row) in enumerate(rows.iterrows()):
    html = coerce_html(row["html"])
    feat = web.get_feature(html)
    if feat:
        samples.append({"track_id": str(i), "html": html, "feature": feat})

clustered, _ = web.cluster_html_struct(samples, threshold=0.95)
dist = Counter(s["layout_id"] for s in clustered)

print(f"50 pages from {host} → {len(dist)} clusters:")
for lid, count in sorted(dist.items(), key=lambda x: -x[1]):
    label = f"cluster {lid}" if lid >= 0 else "noise"
    print(f"  {label:12s}  {'█' * count} ({count})")

In [ ]:
# Visualise the pre-computed global cluster distribution
named = manifest[manifest["dripper_layout_id"].str.startswith("layout-", na=False)]
failed = manifest[~manifest["dripper_layout_id"].str.startswith("layout-", na=False)]
vc = named["dripper_layout_id"].value_counts()

bins = [2, 5, 10, 25, 50, 100, 250, 600]
labels = [f"{bins[i]}-{bins[i + 1] - 1}" for i in range(len(bins) - 1)]
counts = [((vc >= bins[i]) & (vc < bins[i + 1])).sum() for i in range(len(bins) - 1)]
pages = [int(vc[(vc >= bins[i]) & (vc < bins[i + 1])].sum()) for i in range(len(bins) - 1)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(labels, counts, color="steelblue")
axes[0].set(title="Clusters by size", xlabel="Cluster size", ylabel="# clusters")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(labels, pages, color="orange", label="clustered")
axes[1].bar(["failed"], [len(failed)], color="#d9534f", label="no cluster")
axes[1].set(title="Pages by cluster size", xlabel="Cluster size", ylabel="pages")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend()

fig.suptitle(f"{len(named):,} clustered  +  {len(failed):,} failed  =  {len(manifest):,} total", y=1.02)
plt.tight_layout()
plt.show()
print(f"Global clusters: {vc.nunique()}   Ceiling savings: {len(named) / len(manifest) * 100:.1f}%")

## 4. Representative Selection

For each cluster we pick the page with the best **structural coverage** score:
```
score = 0.4 × XPath_coverage + 0.3 × structure_score + 0.3 × width_entropy_score
```
This page is sent to the LLM — all other pages in the cluster are templated from its result.

In [ ]:
biggest_id = vc.index[0]
cluster_df = manifest[manifest["dripper_layout_id"] == biggest_id].head(20)
candidates = [{"track_id": row["url"], "html": coerce_html(row["html"])} for _, row in cluster_df.iterrows()]

rep = web.select_representative_html(candidates)
print(f"Cluster:         {biggest_id}")
print(f"Host:            {cluster_df['url_host_name'].iloc[0]}")
print(f"Cluster size:    {vc[biggest_id]} pages  (showing 20 candidates)")
print(f"Representative:  {rep['track_id'][-80:]}")
print()
print("All candidate URLs:")
for c in candidates:
    marker = " ← SELECTED" if c["track_id"] == rep["track_id"] else ""
    print(f"  {c['track_id'][-80:]}{marker}")

## 5. HTML Simplification

Before the LLM sees the HTML, Dripper simplifies it:
- Removes `<script>`, `<style>`, `<header>`, `<aside>` and non-content structure
- Keeps only `class` and `id` attributes
- Assigns `_item_id="N"` to every remaining node (LLM labels these)
- Truncates long text to first 200 chars per paragraph

Result: **~13% of original** token count — fast and cheap inference.

In [ ]:
def simplify_html(raw, url=""):
    """Returns (simplified_html, mapped_html). Uses the correct stage.py API."""
    case = bindings.case_cls(bindings.input_cls(raw_html=raw, url=url))
    case = bindings.simplify_single_input(case)
    simplified = DripperHTMLExtractionStage._get_processed_attr(case, "simpled_html")
    mapped = DripperHTMLExtractionStage._get_processed_attr(case, "map_html")
    return simplified, mapped


sample_row = manifest[manifest["url_host_name"] == "hysplitbbs.arl.noaa.gov"].iloc[0]
raw = coerce_html(sample_row["html"])

t0 = time.perf_counter()
simp, mapped = simplify_html(raw, url=sample_row["url"])
elapsed = time.perf_counter() - t0

n_items = len(re.findall(r"_item_id=", mapped))
print(f"Page: {sample_row['url']}")
print(f"  Raw HTML:        {len(raw):>8,} chars")
print(f"  Simplified:      {len(simp):>8,} chars  ({len(simp) / len(raw) * 100:.1f}% of original)")
print(f"  Mapped (w/ IDs): {len(mapped):>8,} chars  ({n_items} _item_id nodes)")
print(f"  Time:            {elapsed * 1000:.0f}ms")
print()
print("Simplified HTML (first 500 chars):")
print(simp[:500])

In [ ]:
print("Mapped HTML — each node has _item_id that LLM will label main/other (first 500 chars):")
print(mapped[:500])

## 6. LLM Extraction

The 0.5B model receives the simplified HTML and outputs:  
`{"1": "main", "2": "other", "3": "main", ...}`  

Constrained decoding enforces valid JSON — each item is one of two tokens: `"main"` or `"other"`.

We load responses from the pre-computed baseline (run 328281) instead of re-running the model.

In [ ]:
if baseline is not None:
    merged = manifest.merge(
        baseline[["url", "dripper_prompt_tokens", "dripper_completion_tokens", "dripper_time_s"]], on="url", how="left"
    )
    valid = merged[merged["dripper_prompt_tokens"].notna()]
    print(f"Pages with LLM data: {len(valid):,}")
    print()
    print(valid[["dripper_prompt_tokens", "dripper_completion_tokens", "dripper_time_s"]].describe().round(1))
    total_tok = valid["dripper_prompt_tokens"].sum() + valid["dripper_completion_tokens"].sum()
    print(f"\nTotal tokens: {total_tok:,.0f}  |  Mean inference: {valid['dripper_time_s'].mean():.2f}s/page")

## 7. Template Propagation

Two-step process using the representative's LLM output:

**Step 1 — `map_parser_cls`** (build template)  
Maps the LLM's item labels back to DOM nodes → produces `html_element_dict` (structural template)

Keys: `typical_raw_html`, `typical_raw_tag_html`, `llm_response`

**Step 2 — `layout_parser_cls`** (apply template to sibling)  
Walks sibling's DOM, matches nodes against template, extracts main content — **no GPU call**

Key: `html_source` (sibling HTML) + all fields from step 1

In [ ]:
if baseline is None:
    print("Baseline not loaded — skipping propagation demo.")
else:
    # Find a cluster where we have LLM responses
    merged_full = manifest.merge(
        baseline[["url", "dripper_response", "dripper_content"]].rename(
            columns={"dripper_response": "llm_response", "dripper_content": "llm_content"}
        ),
        on="url",
        how="inner",
    )
    demo_cluster = (
        merged_full[
            merged_full["dripper_layout_id"].str.startswith("layout-", na=False) & merged_full["llm_response"].notna()
        ]
        .groupby("dripper_layout_id")
        .filter(lambda g: len(g) >= 3)
    )

    cid = demo_cluster["dripper_layout_id"].value_counts().index[0]
    cluster = demo_cluster[demo_cluster["dripper_layout_id"] == cid].reset_index(drop=True)
    rep_row = cluster.iloc[0]

    print(f"Demo cluster: {cid}")
    print(f"Host:         {rep_row['url_host_name']}")
    print(f"Pages:        {len(cluster)}  (using first as representative)")
    print(f"Rep URL:      {rep_row['url'][-80:]}")

In [ ]:
if baseline is not None:
    # Step 1: build template from representative
    rep_html = coerce_html(rep_row["html"])
    _, mapped_rep = simplify_html(rep_html, url=rep_row["url"])

    t0 = time.perf_counter()
    template = web.map_parser_cls({}).parse(
        {
            "typical_raw_html": rep_html,
            "typical_raw_tag_html": mapped_rep,
            "llm_response": str(rep_row["llm_response"]),
        }
    )
    map_time = time.perf_counter() - t0

    print(f"Template built in {map_time * 1000:.0f}ms")
    print(f"  typical_main_html_success: {template.get('typical_main_html_success')}")
    print(f"  element_dict depth-0 keys: {list(template.get('html_element_dict', {}).keys())[:5]}")

In [ ]:
if baseline is not None and "template" in dir():
    # Step 2: propagate to sibling — NO GPU
    sibling = cluster.iloc[1]
    sibling_html = coerce_html(sibling["html"])

    task = dict(template)
    task.update(
        {
            "html_source": sibling_html,
            "dynamic_id_enable": True,
            "dynamic_classid_enable": True,
            "more_noise_enable": True,
            "dynamic_classid_similarity_threshold": 0.85,
        }
    )

    t0 = time.perf_counter()
    result = web.layout_parser_cls({}).parse(task)
    prop_time = time.perf_counter() - t0

    prop_html = str(result.get("main_html_body") or "")
    print(f"Propagation in {prop_time:.2f}s  (no GPU!)")
    print(f"  success:  {result.get('main_html_success')}")
    print(f"  sim:      {result.get('main_html_sim'):.3f}" if result.get("main_html_sim") else "  sim: N/A")
    print(f"  output:   {len(prop_html):,} chars")
    print()
    print("Propagated HTML (first 300 chars):")
    print(prop_html[:300])

## 8. Validation — F1 vs Baseline

We compare the propagated content against the pure-LLM baseline using **token-level bag-of-words F1**:  
- Tokenise both strings with `\w+`
- F1 = harmonic mean of precision and recall over token multisets  
- Target: F1 ≥ 0.95 for all propagated rows

In [ ]:
if baseline is not None and "template" in dir():
    f1_rows = []
    for _, row in cluster.iterrows():
        row_html = coerce_html(row["html"])
        t = dict(template)
        t.update(
            {
                "html_source": row_html,
                "dynamic_id_enable": True,
                "dynamic_classid_enable": True,
                "more_noise_enable": True,
                "dynamic_classid_similarity_threshold": 0.85,
            }
        )
        try:
            r = web.layout_parser_cls({}).parse(t)
            prop_html = str(r.get("main_html_body") or "")
            prop_content = convert_to_content(bindings, prop_html, url=str(row.get("url", "")))
        except Exception:
            prop_content = ""

        ref_content = str(row.get("llm_content") or "")
        f1 = _token_f1(prop_content, ref_content)
        f1_rows.append({"url": row["url"], "f1": f1, "prop_len": len(prop_content), "ref_len": len(ref_content)})

    f1_df = pd.DataFrame(f1_rows)
    print(f"F1 results for {len(f1_df)} pages in cluster {cid}:")
    print(f"  Mean F1:   {f1_df['f1'].mean():.4f}")
    print(f"  Min F1:    {f1_df['f1'].min():.4f}")
    print(f"  F1 ≥ 0.95: {(f1_df['f1'] >= 0.95).sum()} / {len(f1_df)}")
    print()
    print(f1_df[["url", "f1", "prop_len", "ref_len"]].to_string(index=False))

if baseline is not None and "template" in dir():
    try:
        from tqdm.notebook import tqdm
    except ImportError:
        from tqdm import tqdm

    MAX_PAGES = 15  # cap for tutorial — propagation is ~11s/page
    sample = cluster.head(MAX_PAGES)
    print(f"Running propagation on {len(sample)} pages (capped at {MAX_PAGES} for speed)")
    print(f"Full cluster has {len(cluster)} pages — ~{len(cluster)*11/60:.0f} min to do all")
    print()

    f1_rows = []
    t_total = time.perf_counter()

    for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Propagating"):
        row_html = coerce_html(row["html"])
        t = dict(template)
        t.update({"html_source": row_html, "dynamic_id_enable": True,
                  "dynamic_classid_enable": True, "more_noise_enable": True,
                  "dynamic_classid_similarity_threshold": 0.85})
        t0 = time.perf_counter()
        try:
            r = web.layout_parser_cls({}).parse(t)
            prop_html    = str(r.get("main_html_body") or "")
            prop_content = convert_to_content(bindings, prop_html, url=str(row.get("url", "")))
            elapsed = time.perf_counter() - t0
            success = r.get("main_html_success", False)
        except Exception as e:
            prop_content = ""
            elapsed = time.perf_counter() - t0
            success = False

        ref_content = str(row.get("llm_content") or "")
        f1 = _token_f1(prop_content, ref_content)
        f1_rows.append({"url": row["url"], "f1": f1,
                        "prop_len": len(prop_content), "ref_len": len(ref_content),
                        "time_s": elapsed, "success": success})

    wall = time.perf_counter() - t_total
    f1_df = pd.DataFrame(f1_rows)

    print(f"\nDone in {wall:.1f}s  ({wall/len(sample):.1f}s/page avg)")
    print(f"\nF1 distribution across {len(f1_df)} pages:")
    print(f"  Mean F1:   {f1_df['f1'].mean():.4f}")
    print(f"  Min F1:    {f1_df['f1'].min():.4f}")
    print(f"  F1 ≥ 0.95: {(f1_df['f1'] >= 0.95).sum()} / {len(f1_df)}")
    print(f"  Succeeded: {f1_df['success'].sum()} / {len(f1_df)}")
    print()
    print(f1_df[["url", "f1", "time_s", "prop_len", "ref_len"]].to_string(index=False))

In [ ]:
total = len(manifest)
named_v = manifest[manifest["dripper_layout_id"].str.startswith("layout-", na=False)]
vc2 = named_v["dripper_layout_id"].value_counts()
n_clust = len(vc2)
standalone = total - len(named_v)
rep_calls = n_clust  # 1 LLM call per cluster (representative)
val_calls = n_clust * 2  # 2 validation LLM calls per cluster
propagated = len(named_v) - rep_calls - val_calls
total_llm = rep_calls + val_calls + standalone
reduction = 1 - total_llm / total

print("=" * 55)
print("COST ANALYSIS — 8192 pages, CC-MAIN-2025-26")
print("=" * 55)
print(f"Total pages:          {total:>6,}")
print()
print("Pure Dripper (baseline):")
print(f"  LLM calls:          {total:>6,}  (every page)")
print("  Projected H100h:    241,993")
print()
print("Layout template mode:")
print(f"  Clusters:           {n_clust:>6,}")
print(f"  Representative LLM: {rep_calls:>6,}")
print(f"  Validation LLM:     {val_calls:>6,}")
print(f"  Standalone LLM:     {standalone:>6,}")
print(f"  Propagated (CPU):   {propagated:>6,}  ← no GPU")
print(f"  Total LLM calls:    {total_llm:>6,}")
print(f"  Theoretical saving: {reduction * 100:.1f}%")
print()
print("Measured (run 330654, best validated config):")
print("  Actual call reduction: 26.0%")
print("  Saved rows mean F1:    0.9871")
print("  Projected H100h:       387,447")
print()
print("With deferred propagation (job 332432, in progress):")
print("  GPU stage removes ~24,000s CPU propagation")
print("  Projected H100h:       ~160,000")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
configs = ["Pure Dripper\n(baseline)", "Layout+Validation\n(best measured)", "Deferred Propagation\n(in progress)"]
h100h = [241993, 387447, 160000]
colors = ["#d9534f", "#f0ad4e", "#5cb85c"]
bars = ax.bar(configs, h100h, color=colors, width=0.5, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, h100h):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 3000,
        f"{val:,}",
        ha="center",
        fontsize=10,
        fontweight="bold",
    )
ax.set_ylabel("Projected H100-hours (full CC snapshot)")
ax.set_title("Dripper Cost Reduction — CC-MAIN-2025-26 (~2.4B pages)")
ax.set_ylim(0, 500000)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x / 1000:.0f}K"))
plt.tight_layout()
plt.show()